In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
import ipywidgets as widgets
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)

In [2]:
map = geemap.Map()

point = ee.Geometry.Point([90.4152, 23.8041]) #around dhaka
region = ee.Geometry.Rectangle([90.3, 23.7, 90.5, 23.9]) #bounding box around the point

top_left = [23.822424724001266, 90.46289150228976]
bottom_right = [23.796369273111445, 90.5071668854189]

#region = ee.Geometry.Rectangle([top_left[1], bottom_right[0], bottom_right[1], top_left[0]])
region = ee.Geometry.Rectangle([92.12743533806102, 21.81144778755447, 92.56414188102977, 22.207414604160263])

map.centerObject(region, 10)
map.add_basemap('HYBRID')
map.addLayer(point, {'color': 'red'}, 'Point Layer')
map.addLayer(region, {'color': 'blue'}, 'Region Layer')


map.default_style = {'cursor': 'crosshair'}


map #just testing if everything is working

Map(center=[22.009483577149734, 92.34578860954535], controls=(WidgetControl(options=['position', 'transparent_…

In [6]:
#now loading the elevation data - NASADEM

dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation').clip(region)

hillshade = ee.Terrain.hillshade(dem, 315, 45)
'''so what hillshade does is it simulates the effect of sunlight on the tarraine, giving a 3D appearance to the 2D map.
this is very useful for visualizing elevation changes and landforms.'''


contours = ee.Image(0).mask(dem.mod(50).lt(2))
'''contour lines are created (like we usually see in topographic maps) at a regular interval to give us the idea of 
elevation changes'''

slope = ee.Terrain.slope(dem)
steepRisk = slope.gt(20).selfMask()

In [17]:
Map = geemap.Map()

Map.addLayer(hillshade, {'min': 0, 'max': 255}, 'hillshade')
visParams = {
  'min': 0, 
  'max': 300, 
  'palette': ['green', 'yellow', 'brown', 'white']
}
Map.addLayer(dem, visParams, 'elevation', True, 0.5)
Map.addLayer(steepRisk, {'palette': 'red'}, 'steep slopes >20°', False)
# dem_rounded = dem.round()
# contours = dem_rounded.mod(50).eq(0).selfMask()
# Map.addLayer(contours, {'palette': 'black'}, 'contours every 50m', True, 0.5)
contours = geemap.create_contours(dem, 0, 5000, 50, region=region)
Map.addLayer(contours, {'palette': 'black'}, 'contours every 100m', True, 0.5)
Map.centerObject(region, 10)
Map

Map(center=[22.009483577149734, 92.34578860954535], controls=(WidgetControl(options=['position', 'transparent_…